In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.7 MB/s eta 0:00:00


In [7]:
import numpy as np
import pandas as pd
import faiss
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
embeddings = np.load("/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings/clip_embeddings.npy")
metadata = pd.read_csv("/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/metadata.csv")
index = faiss.read_index("/content/drive/MyDrive/copydays-ndid/Nagarjuna/ndid_faiss.index")

embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [8]:
THRESHOLD = 0.85

y_true = []
y_pred = []

for i in tqdm(range(len(embeddings))):

    query_embedding = embeddings[i].reshape(1, -1)

    D, I = index.search(query_embedding, 2)

    # ignore self-match
    best_idx = I[0][1]
    best_score = D[0][1]

    true_group = metadata.iloc[i]["group_id"]
    pred_group = metadata.iloc[best_idx]["group_id"]

    is_duplicate = best_score >= THRESHOLD

    y_true.append(True)  # because dataset is near-duplicate
    y_pred.append(is_duplicate and (true_group == pred_group))


100%|██████████| 30000/30000 [04:14<00:00, 117.72it/s]


In [9]:
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 1.0
Recall: 0.9914333333333334
F1 Score: 0.9956982407980851
